# 05 — Interventions

Interactive analysis of Phase 4 results: detection classifiers and steering interventions.

Goals:
- Probe layer sweep: which layer gives the best classifier accuracy for each failure mode?
- Compare SAE feature direction vs. DiffMean steering across multipliers
- Plot faithfulness/sycophancy improvement vs. accuracy degradation (capability trade-off)
- Identify the best intervention configuration for each failure mode
- Error analysis: on which prompt types does each intervention fail?

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PHASE4_RUN = 'FILL_IN_RUN_ID'
PHASE4_DIR = Path('../experiments/results/phase4')

In [ ]:
# Load Phase 4 results
with open(PHASE4_DIR / PHASE4_RUN / 'classifier_results.json') as f:
    clf_results = json.load(f)
with open(PHASE4_DIR / PHASE4_RUN / 'steering_results.json') as f:
    steering_results = json.load(f)
with open(PHASE4_DIR / PHASE4_RUN / 'run_summary.json') as f:
    summary = json.load(f)

print('=== Classifier results ===')
for target, r in clf_results.items():
    print(f"{target}: layer={r['best_layer']}, CV accuracy={r['cv_accuracy']:.3f}")

In [ ]:
# Compare DiffMean vs SAE feature direction steering
# Focus on sycophancy task
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

tasks = [
    ('sycophancy', 'sycophancy_rate', 'Sycophancy rate (lower=better)', axes[0]),
    ('faithfulness', 'faithfulness_score', 'Faithfulness score (higher=better)', axes[1]),
]

for task, metric_key, ylabel, ax in tasks:
    for method in ['diffmean', 'sae_feature_direction']:
        key = f'{method}_{task}'
        if key not in steering_results:
            continue
        results = steering_results[key]
        multipliers = [r['multiplier'] for r in results]
        metric_vals = [r[metric_key] for r in results]
        accuracy = [r['accuracy'] for r in results]
        
        color = '#2166ac' if method == 'sae_feature_direction' else '#d6604d'
        label = 'SAE feature direction (ours)' if method == 'sae_feature_direction' else 'DiffMean (Rimsky et al.)'
        ax.plot(multipliers, metric_vals, 'o-', color=color, label=label, linewidth=1.5)
    
    ax.set_xlabel('Steering multiplier')
    ax.set_ylabel(ylabel)
    ax.set_title(f'{task.title()} task')
    ax.legend(fontsize=8)
    ax.axvline(0, color='grey', linestyle='--', linewidth=0.8)

plt.suptitle('Steering method comparison', fontsize=11)
plt.tight_layout()
plt.savefig('../paper/figures/steering_comparison.png', dpi=150)
plt.show()

In [ ]:
# Capability trade-off: improvement in target metric vs. accuracy loss
fig, ax = plt.subplots(figsize=(8, 6))

colors = {'diffmean_sycophancy': '#f4a582', 'sae_feature_direction_sycophancy': '#d6604d',
          'diffmean_faithfulness': '#92c5de', 'sae_feature_direction_faithfulness': '#2166ac'}

for key, results in steering_results.items():
    if not results:
        continue
    # Use the most effective multiplier (highest improvement without large accuracy loss)
    task = 'sycophancy' if 'sycophancy' in key else 'faithfulness'
    metric = 'sycophancy_rate' if task == 'sycophancy' else 'faithfulness_score'
    
    for r in results:
        # For sycophancy: improvement = baseline - new rate; for faithfulness: improvement = new - baseline
        if task == 'sycophancy':
            improvement = -r[metric]  # Lower sycophancy is better
        else:
            improvement = r[metric]
        acc_delta = r['accuracy']
        
        ax.scatter(acc_delta, improvement, color=colors.get(key, 'grey'), alpha=0.7, s=60)

# Add legend
for key, color in colors.items():
    ax.scatter([], [], color=color, label=key, s=60)

ax.axvline(0, color='grey', linestyle='--', linewidth=0.8)
ax.axhline(0, color='grey', linestyle='--', linewidth=0.8)
ax.set_xlabel('Accuracy delta (positive = preserved capability)')
ax.set_ylabel('Target metric improvement')
ax.set_title('Intervention trade-off: effectiveness vs. capability cost')
ax.legend(fontsize=8, loc='lower right')
# Top-right quadrant = effective and capability-preserving
ax.annotate('Ideal region', xy=(0.01, 0.01), xycoords='axes fraction',
            fontsize=8, color='grey', ha='left', va='bottom')

plt.tight_layout()
plt.savefig('../paper/figures/intervention_tradeoff.png', dpi=150)
plt.show()

In [ ]:
# Best configuration table
rows = []
for key, results in steering_results.items():
    if not results:
        continue
    task = 'sycophancy' if 'sycophancy' in key else 'faithfulness'
    method = 'SAE direction' if 'sae_feature' in key else 'DiffMean'
    
    # Best = maximize target improvement while keeping accuracy within 3%
    metric = 'sycophancy_rate' if task == 'sycophancy' else 'faithfulness_score'
    eligible = [r for r in results if abs(r['accuracy']) <= 0.03]
    if not eligible:
        eligible = results  # fallback: no accuracy constraint
    
    if task == 'sycophancy':
        best = min(eligible, key=lambda r: r[metric])
        improvement = best[metric]  # show final rate
    else:
        best = max(eligible, key=lambda r: r[metric])
        improvement = best[metric]
    
    rows.append({
        'Method': method, 'Task': task,
        'Multiplier': best['multiplier'], 'Layer': best['layer'],
        metric: f"{improvement:.3f}",
        'Accuracy': f"{best['accuracy']:.3f}",
        'Over-correction': f"{best['over_correction_rate']:.3f}",
    })

print(pd.DataFrame(rows).to_string(index=False))